<a href="https://colab.research.google.com/github/ironspiritjeff/hf-transformer-experiments/blob/main/HuggingFace_TextClassificationTutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning Hugging Face Text Classification Tutorial

* Resource Notebook: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial
* Setup steps:https://www.learnhuggingface.com/extras/setup

Note: A GPU is required

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
HF_TOKEN

## Importing necessary libraries

In [ ]:
# Install dependencies
try:
  import datasets, evaluate, accelerate
  import gradio
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio
  import datasets, evaluate, accelerate
  import gradio as gr

import random # Ensure random is imported
import numpy as np
import pandas as pd # Ensure pandas is imported

import torch
import transformers

print(f'Using transformers version: {transformers.__version__}')
print(f'Using torch version: {torch.__version__}')
print(f'Using datasets version: {datasets.__version__}')

## Getting a dataset

Food Not Food data

In [ ]:
from datasets import load_dataset

dataset = load_dataset('mrdbourke/learn_hf_food_not_food_image_captions')
dataset

In [ ]:
# What features are there?
dataset.column_names

In [ ]:
# Access the training split
dataset['train']

In [ ]:
# Look at a random sample
dataset['train'][10]

In [ ]:
### Inspect random samples
import random

random_indices = random.sample(range(len(dataset['train'])), 5)
print(random_indices )

random_samples = dataset['train'][random_indices]

print(f'[INFO] Random samples from dataset:\n')
for text, label in zip(random_samples['text'], random_samples['label']):
  print(f'Text: {text} | Label: {label}')

In [ ]:
# Get unique label values
dataset['train'].unique('label')

In [ ]:
# Check the count of each label
from collections import Counter

Counter(dataset['train']['label'])

In [ ]:
# Turn dataset into a DataFrame and get a random sample
food_not_food_df = pd.DataFrame(dataset['train'])
food_not_food_df.sample(7)

In [ ]:
food_not_food_df['label'].value_counts()

## Preparing data for text classification

* Tokenize our text - turn it into numbers.
* Split the data into train and test splits.

In [ ]:
# Create a mapping for labels to numeric value (by hand) - CONSISTENT MAPPING
id2label = {0: 'not_food', 1: 'food'} # 0 = not_food, 1 = food
label2id = {'food': 1, 'not_food': 0} # food = 1, not_food = 0

print(id2label)
print(label2id)

In [ ]:
# Create mappings programmatically from dataset
# id2label = {idx: label for idx, label in enumerate(dataset['train'].unique('label'))}
# label2id = {label: idx for idx, label in id2label.items()}
# print(id2label)
# print(label2id)

In [ ]:
# Turn labels into 0 or 1
def map_labels_to_number(example):
  example['label'] = label2id[example['label']]
  return example

example_sample = {'Text': 'This is about blueberries', 'label': 'food'}

map_labels_to_number(example_sample)

In [ ]:
# This cell caused a KeyError. The proper re-mapping is handled in cell 73ce8ab7 by reloading the original dataset.
# We will explicitly skip re-mapping here to avoid issues.
# dataset = dataset['train'].map(map_labels_to_number)
# dataset[:5]

print("Cell mZn3TeqiL_sE skipped for consistent mapping. Proceed to cell dd280ed3 or 73ce8ab7.")

### Reverse the label mapping

In [ ]:
# Ensure id2label and label2id are consistent with the desired mapping - CONSISTENT MAPPING
id2label = {0: 'not_food', 1: 'food'}
label2id = {'food': 1, 'not_food': 0}

print(id2label)
print(label2id)

Now that `label2id` is updated, we need to re-apply the mapping function to the dataset. Note that this will overwrite the previous numerical labels.

In [ ]:
# Re-map our dataset labels to numbers with the new mapping
# Since the dataset was already mapped, we need to reload it or ensure the original 'label' column is still text.
# For simplicity, let's assume you want to re-run the entire process from loading the dataset if you make a mistake.
# However, if you just want to reverse the current numerical labels, you could also define a function to swap 0s and 1s.

# To safely re-apply with the new mapping, it's often best to re-load the dataset to ensure labels are text again
from datasets import load_dataset
original_dataset = load_dataset('mrdbourke/learn_hf_food_not_food_image_captions')
dataset = original_dataset['train'].map(map_labels_to_number)
dataset[:5]

In [ ]:
# Shuffle data and look at 5 more random samples
dataset.shuffle()[:5]

### Split data into train and test splits
HF has a function for this: datasets.Dataset.train_test_split

In [ ]:
dataset = dataset.train_test_split(test_size = 0.2, seed = 42)
dataset

In [ ]:
# Ensure 'random' is imported globally or locally if this cell is run out of sequence
import random
random_idx_train = random.randint(0, len(dataset['train']))
random_sample_train = dataset['train'][random_idx_train]
random_sample_train

In [ ]:
# Ensure 'random' is imported globally or locally if this cell is run out of sequence
import random
random_idx_test = random.randint(0, len(dataset['test']))
random_sample_test = dataset['test'][random_idx_test]
random_sample_test

### Tokenizing our text data
The Hugging Face  `transformers` library has built-in support for Hugging Face `tokenizers`.

And the `transformers.AutoTokenizer` helps pair a model to a tokenizer.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path='distilbert/distilbert-base-uncased',
                                          use_fast=True) #use the fast implimentation,
                                          # note: this requires Rust installed
tokenizer

In [ ]:
# Test out the tokenizer
tokenizer('extraordinary')

## Tokenization

* `input_ids` = our text turned into numbers
* `attention mask` = whether orr not to pay attenion to certain tokens (1 = yes pay attention, = no don't pay attention)

The `transformers` library has in-built support for Hugging Face `tokenizers`.

And the class `transformers.AutoTokenizer` helps pair a model to a tokenizer.


The tokenizer we're using is distilbert: https://huggingface.co/distilbert/distilbert-base-uncased

* Models are often paired with tokenizers
  * We can create tokenizer/model instances with HuggingFace's `Auto` classes:
  https://huggingface.co/docs/transformers/en/model_doc/auto
* Tokenizer = turn text into numbers
* Model = finds patterns in those numbers

In [ ]:
# Get the length of our tokenizer vocab
length_tok_vocab = len(tokenizer.vocab)
print(f'[INFO] Number of items in our tokenizer vocab: {length_tok_vocab}')

# Get the maximum sequence length the tokenizer can handle
max_tokenizer_input_sequence_length = tokenizer.model_max_length
print(f'[INFO] Max tokenizer input sequence length: {max_tokenizer_input_sequence_length}')

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer('❤️').input_ids)

In [ ]:
# Get the first 5 items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:5]

In [ ]:
# Ensure 'random' is imported globally or locally if this cell is run out of sequence
import random
random.sample(sorted(tokenizer.vocab.items()), k=5)

### Create a preprocessing function to tokenize our text

We want to make it easy to go from sample to tokenized sample.

In [ ]:
def tokenize_text(examples):
  '''
  Tokenize the given example text and return the tokenized text
  '''

  return tokenizer(examples['text'],
            padding=True, # pad short sequences to longest length in batch.
            # (eg. if sample length =100, sample will be padded to 512 or longest sample in batch)
            truncation=True) # truncates long sequences to the maximum length the model can handle
            # (eg if the sample length = 1000, if model length = 512, sample will be shortened to 512)

In [ ]:
# Map our tokenized text function to the dataset
tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True,
                                batch_size=1000)

# Set batches=True to tokenize across batches of samples at a time rather than one at a time

* **Note** In machine learning, it's often faster to do things in batches rather than one at a time,  due to leveraging computer hardware parallelization in computing.
See more in the `process` documentation https://huggingface.co/docs/datasets/process

In [ ]:
# Get two samples from the tokenized datasets
train_tokenized_sample = tokenized_dataset['train'][0]
test_tokenized_sample = tokenized_dataset['test'][0]

for key in train_tokenized_sample.keys():
  print(f'[INFO] Key: {key}')
  print(f'Train sample: {train_tokenized_sample[key]}')
  print(f'Test sample: {test_tokenized_sample[key]}')
  print()

### Tokenization takeaways
1. Tokenizers = turn data into numbers (eg. text -> map to number)
2. Many models are out there and have different tokenizers; Hugging Face's `Auto` (eg. `Autotokenizer`, `AutoProcessor`, `AutoModel` etc helps to match tokenizers to models).
3. Tokenization can happen in parallel using map and batched functions.


## Setting up an evaluation metric

We want to use the evaluation metric to get an numerical idea of how our model is performing.

Some common evaluation metrics for classification:
- accuracy (how many predictions out of 100 were predicted correctly)
- Precision
- Recall
- F1 Score

Evaluation metric is important because some projects may have an evaluation threshold you need to fulfill. (Eg. insurance claim classification 98%+ test accuracy to be commercially viable)

Some places for evaluation metrics:
* scikit-learn
* hugging face evaluate: https://huggingface.co/docs/evaluate/en/index

In [ ]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load('accuracy')
def compute_accuracy(predictions_and_labels: Tuple[np.array, np.array]):
  '''
  Computes the accuracy of a model by computing the predictions and labels.
  '''
  predictions, labels= predictions_and_labels

  if len(predictions.shape) >=2:
    predictions=np.argmax(predictions, axis=1)

  return accuracy_metric.compute(predictions = predictions, references=labels)

In [ ]:
# Example predictions and accuracy score
example_preds_all_correct = np.array([0,0,0,0,0,0,0,0,0,0])
example_preds_one_incorrect = np.array([0,0,0,0,0,1,0,0,0,0])
example_labels = np.array([0,0,0,0,0,0,0,0,0,0])

# Test the function
print(f'Accuracy when all predictions are correct: {compute_accuracy((example_preds_all_correct, example_labels))}')
print(f'Accuracy when one prediction is incorrect: {compute_accuracy((example_preds_one_incorrect, example_labels))}')

# Model Training
## Setting up training arguments (hyperparameters) with TrainingArguments

Transfer learning is a technique for reusing knowledge learned from a task to boost performance on a related task.

Workflow for training:
1. Create and preprocess data
2. Choose the model we'd like to use for our problem
  * from huggingface.co/models
  * Or see "task guides" in HF Transformers docs: huggingface.co/docs/transformers/en/tasks/sequence_classification
3. Define training arguments for training our model `transformers.TrainingArguments`
  * These are known as "hyperparameters" = settings on your model that you can adjust
  * Parameters = weights/patterns in the model that get updated automatically
  * https://huggingface.co/docs/transformers/main_classes/trainer to see all training arguments
4. Pass `TrainingArguments` to an instance of `transformers.Trainer`
5. Train the model by calling `Trainer.train()`
6. Save the model (to our local machine or to Hugging Face Hub)
7. Evaluate the trained model by making and inspecting predictions on the test data (and our own custom data!)
8. Turn the model into a shareable demo


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
  pretrained_model_name_or_path='distilbert/distilbert-base-uncased',
  num_labels=2, #classify into food/not food
  id2label = id2label,
  label2id=label2id)

In [ ]:
model

Our model is comprised of the following parts:

1. `embeddings` - embeddings are a form of learned representation of tokens. So if tokens are a direct maping from token to number, embeddings are a learned vector representation.
2. `transformer` - our model architecture backbone; this has discovered patterns/relationships in the embeddings.
3. `classifier` - we need to customize this later to suit our problem.


Note: If you get input errors from passing a sample to a model, make sure the sample you pass to your model is formatting in the same way your model was trained on. For example, if your model used a specific tokenizer, make sure to tokenize your text before passing it to the model.

### Count parameters in our model

Weights/parameters = small numeric opportunities for a model to learn patterns in data.


In [ ]:
def count_params(model):
  '''
  Count the parameters of a PyTorch model.
  '''
  trainable_parameters = sum(param.numel() for param in model.parameters() if param.requires_grad)
  total_parameters = sum(param.numel() for param in model.parameters())

  return {'trainable_parameters': trainable_parameters, 'total_parameters': total_parameters}

count_params(model)

Our model has around 67 million parameters and **all** of them are trainable.

Note:
* Generally the more parameters a model has, the more capacity it has to learn.
* For comparison, models such as Llama 3 8B has 8 billion parameters.
* However, more parameters requires more compute + time. Smaller models can still be surprisingly effective.

### Create a directory for saving models

In [ ]:
#  Create model output directory
from pathlib import Path

# Create models dir
models_dir = Path('models')
models_dir.mkdir(exist_ok=True)

# Create model save name
model_save_name = 'learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

# Create model save path
model_save_dir = Path(models_dir, model_save_name)

model_save_dir

In [ ]:
from transformers import TrainingArguments

print(f'[INFO] Saving model checkpoints: {model_save_dir}')

BATCH_SIZE=32

# Create training arguments
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    use_cpu=False,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy='epoch',
    report_to='none',
    # hub_token= ...
    # push_to_hub=True
    hub_private_repo=False #When uploading to HF Hub, do you want your repo to be private or public?
)

In [ ]:
# training_args
# (to see full list of both the ones we set, and the default arguments)

## 4. Setting up an instance of Trainer

* Docs for Transformer.Trainer
https://huggingface.co/docs/transformers/main_classes/trainer

* Our model is going to expect the tokenized version of our dataset, including input ids and attention mask.

In [ ]:
from transformers import Trainer


trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    compute_metrics=compute_accuracy
)
trainer

5. Train the model using trainer.train

In [ ]:
results = trainer.train()

In [ ]:
results.metrics

### Save the model for later use.
(right away with colab)

In [ ]:
# Save model
print(f'[INFO] Saving model to {model_save_dir}')
trainer.save_model(output_dir=model_save_dir)

### Inspect the model training metrics

In [ ]:
# Get training history
trainer_history_all = trainer.state.log_history
trainer_history_metrics = trainer_history_all[:-1]

trainer_history_training_time = trainer_history_all[-1]

#View the first 3
trainer_history_metrics[:3]

In [ ]:
import pprint

# Extract eval and training metrics
trainer_history_training_set = []
trainer_history_eval_set = []

# Loop through our metrics
for item in trainer_history_metrics:
  item_keys = list(item.keys())
  if any('eval' in item for item in item_keys):
    trainer_history_eval_set.append(item)
  else: trainer_history_training_set.append(item)

# Show the first item from each
print(f'First item in training set:')
pprint.pprint(trainer_history_training_set[:2])

print('\nFirst two items in eval epochs:')
pprint.pprint(trainer_history_eval_set[:2])

### Taking a look at the loss curves

Loss curves = a good visualization of model's perfomance over time

In [ ]:
# Create a Pandas DataFrame for the training and evaluation metrics
trainer_history_train_df = pd.DataFrame(trainer_history_training_set)
trainer_history_eval_df = pd.DataFrame(trainer_history_eval_set)

trainer_history_train_df.head()

In [ ]:
# Plot the loss curves
import matplotlib.pyplot as plt

plt.figure(figsize=(5,3))
plt.plot(trainer_history_train_df['epoch'], trainer_history_train_df['loss'], label='Training Loss')
plt.plot(trainer_history_eval_df['epoch'], trainer_history_eval_df['eval_loss'], label='Eval loss')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Text classification fine tuning DistilBert training and evaluation loss over time')
plt.legend()
plt.show()

### Pushing our model to the HF Hub

Why do this?
* So we can share our model
* Other people can try them out
* We can keep a history of different model versions

To write to Hugging Face:
 * If on Google Colab: set up 'token' with 'read and write' access
 * If on local machine: set up `huggingface-cli` https://huggingface.co/docs/huggingface_hub/en/guides/cli

To save to the Hugging Face Hub, use the `Trainer.push_to_hub` method

In [ ]:
# Save our model to the Hugging Face Hub
model_upload_url = trainer.push_to_hub(
    commit_message='Uploading food not food text classifier model',
    )
print(f'[INFO] Model successfully uploaded to the Hugging Face Hub with URL: {model_upload_url}')

## Making and Evaluating predictions on the test data

**Note** Evaluating a model is just as important as training a model.

We can make predictions on the test data using the `Trainer.predict` method.  https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer.predict

Remember that a model has to be tested on the same format of data it was trained on. The model needs to make predictions on data in the same format /distribution it was trained on.

In [ ]:
tokenized_dataset['train']

In [ ]:
tokenized_dataset['test']

Perform predictions on the test data

In [ ]:
predictions_all = trainer.predict(tokenized_dataset['test'])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f'[INFO] Prediction metrics on the test data:')
prediction_metrics

In [ ]:
# View  predictions array. The higher number represents the prediction
predictions_all

### Let's get predicted probablilities and evaluate by hand.

Use PyTorch to take raw outputs ("predicted logits") and turn them into prediction probabilities with torch.softmax -> predicted labels

* Softmax gets all values to be between 0-1, and the total of the values to sum to 1.
* The new values are what's known as a prediction probablitity. These reflect the model's confidence, not how right it actually is.

In [ ]:
print(f'Raw label: {predictions_all.predictions[1]}')
print(f'\nPrediction probability with Softmax:\n {torch.softmax(torch.tensor(predictions_all.predictions[0]), dim=0)}')

In [ ]:
import torch
from sklearn.metrics import accuracy_score

# 1. Get prediction probabilities with torch.softmax
pred_probs = torch.softmax(torch.tensor(prediction_values), dim=1)

# 2. Get the predicted labels
pred_labels = torch.argmax(pred_probs,dim=1)

# Get the true labels
true_labels = tokenized_dataset['test']['label']

# Compute prediction labels to true labels and get the test accuracy
test_accuracy = accuracy_score(y_true = true_labels,
                               y_pred = pred_labels)
print(f'[INFO] Test accuracy: {test_accuracy*100}')

Note: if you want a good evaluation method, make predictions on your entire test dataset, then index on the predictions which are wrong but have high prediction probability.
* For example, get the top 100-1000 and go through all of the examples where the model's prediction had high probability but was incorrect -> This often leads to great insights into your data.

### Exploring our model's prediction probabilities

Evaluate a model by sorting pred probabilities and seeing where it went wrong.

In [ ]:
# Make a DataFrame of test predictions
test_predictions_df = pd.DataFrame({
    'text': tokenized_dataset['test']['text'],
    'true_label': true_labels,
    'pred_label': pred_labels,
    'pred_prob': torch.max(pred_probs, dim=1).values
})
test_predictions_df.head()

In [ ]:
# Show 10 examples with low prediction probability
test_predictions_df.sort_values('pred_prob', ascending=True).head(10)

# Performing Inference

In [ ]:
# Set up device for making predictions

def set_device():
  if torch.cuda.is_available():
    device = torch.device('cuda')
  elif torch.backends.mps.is_available() and torch.bbackends.mps.is_built():
    device = torch.device('mps')
  else:
    device= torch.device('cpu')
  return device

DEVICE = set_device()
print(f'[INFO] Using device: {DEVICE}')

## Making and inspecting predictions on custom data.

Note: inference = using a model to make predictions on data.

Two main ways to perform inference:
1. **Pipeline mode** - Using `transformers.pipeline` to load our model and perform text classification. See the docs: https://huggingface.co/docs/transformers/en/main_classes/pipelines
2. **Pytorch mode** - Using a combination of `transformers.AutoTokenizer` and `transformers.AutoModelForSequenceClassification` and passing each our target model name.

Each mode supports:
1. Predictions one at a time ( fast, but can be slower with large numbers of samples)
* Eg. helpful for a comment system when comments happen sporadically, to predict whether it was "spam" or "not spam".
2. Batches of predictions at a time (faster but up to a point; eg say you predict on 32 samples at a time, this may be way faster than one at a time but if you go to 128 at a time, you may not see many more speedups)
* Helpful for when you have a large static database of many samples coming in at once.

In [ ]:
local_model_path = 'models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

huggingface_model_path = 'ironspiritjeff/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

In [ ]:
import torch
from transformers import pipeline

# Set the batch size for predictions
BATCH_SIZE = 32

# Create an instance of transformers.pipeline
food_not_food_classifier = pipeline(task="text-classification", # we can use this because our model is an instance of AutoModelForSequenceClassification
                                    model=local_model_path, # could also pass in huggingface_model_path
                                    device=DEVICE, # set the target device
                                    top_k=1, # only return the top predicted value
                                    batch_size=BATCH_SIZE) # perform predictions on up to BATCH_SIZE number of samples at a time

food_not_food_classifier

In [ ]:
food_not_food_classifier('avocados sliced on toast')

In [ ]:
# Use Pipeline with model from Hugging Face
from transformers import pipeline
food_not_food_classifier2 = pipeline(task='text-classification',
                                     model=huggingface_model_path,
                                     device=DEVICE,
                                     top_k=1,
                                     batch_size=BATCH_SIZE)
food_not_food_classifier2

In [ ]:
food_not_food_classifier2('pineapple and chocolate')

### Making multiple predictions at the same time with batch predictions

In [ ]:
# Create a list of sentences to make predictions on
sentences = [
    "I whipped up a fresh batch of code, but it seems to have a syntax error.",
    "We need to marinate these ideas overnight before presenting them to the client.",
    "The new software is definitely a spicy upgrade, taking some time to get used to.",
    "Her social media post was the perfect recipe for a viral sensation.",
    "He served up a rebuttal full of facts, leaving his opponent speechless.",
    "The team needs to simmer down a bit before tackling the next challenge.",
    "The presentation was a delicious blend of humor and information, keeping the audience engaged.",
    "A beautiful array of fake wax foods (shokuhin sampuru) in the front of a Japanese restaurant.",
    "strawberries",
    "My favoruite food is pizza and chocolate!"
]

food_not_food_classifier(sentences)

### Time our model across larger samples

In [ ]:
import time

# Create 1000 samples
sentences_1000 = sentences*100

# Time how long it takes to make predictions on all sentences (one at a time)
print(f'Number of sentences: {len(sentences_1000)}')
start_time_one_at_a_time = time.time()
for sentence in sentences_1000:
  #Make a prediction
  food_not_food_classifier2(sentence)
end_time_one_at_a_time = time.time()

total_time_one_at_a_time = end_time_one_at_a_time - start_time_one_at_a_time
avg_time_per_pred = total_time_one_at_a_time / len(sentences_1000)
print(f'[INFO] Total time for making predictions on {len(sentences_1000)}: {total_time_one_at_a_time}s')
print(f'[INFO] Avg time per prediction: {avg_time_per_pred}s')

In [ ]:
# Let's now use pipeline in batches
for i in [10, 100, 200]:
  sentences_big = sentences *i
  print(f'[INFO] Number of sentences: {len(sentences_big)}')

  start_time = time.time()
  # Predict on all sentences in batch mode
  food_not_food_classifier2(sentences_big)
  end_time = time.time()

  total_time_per_all_sentences_batch_mode = end_time - start_time
  avg_time_per_sentence_batch_mode = total_time_per_all_sentences_batch_mode / len(sentences_big)

  print(f'[INFO] Inference time for {len(sentences_big)} sentences batch mode: {round(total_time_per_all_sentences_batch_mode, 6)}')
  print(f'[INFO] Avg inference time per sentence: {round(avg_time_per_sentence_batch_mode, 8)}')


### Making predictions with PyTorch
(for more customization)

Steps with PyTorch predictions:
1. Create the tokenizer with `AutoTokenizer`
2. Create the model with `AutoModel` (`AutoModelForSequenceClassification`)
3. Tokenize the text with 1
4. Make predictions with 2
5. Format prediction

In [ ]:
from transformers import AutoTokenizer
# Set up the model path
model_path = huggingface_model_path

# Create an example to predict on
sample_food_text = 'A delicious photo of screambled eggs, toast, and bacon'

# Prepare the tokenizer
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path = model_path)
inputs = tokenizer(sample_food_text,
                   return_tensors='pt') # pt stands for pytorch, which defaults to tensors

inputs

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=model_path)
model

In [ ]:
import torch
model.eval()
# With torch.no_grad():
with torch.inference_mode():
  outputs = model(**inputs) # ** means input all of the dictionary keys as named arguments/parameters
  outputs_verbose = model(input_ids=inputs['input_ids'],
                          attention_mask=inputs['attention_mask'])
  outputs

In [ ]:
# Convert logits to prediction probability +label
predicted_class_id = outputs.logits.argmax().item()
prediction_probability = torch.softmax(outputs.logits, dim=1).max().item()

print(f'Text: {sample_food_text}')
print(f'Predicted label:{model.config.id2label[predicted_class_id]}')
print(f'Prediction probability: {prediction_probability}')

# Putting It All Together
End-to-end; from data import to model building to model evaluation to model saving for our text classification project.

In [1]:
# 1. Import necessary packages
import pprint
from pathlib import Path
import numpy as np
import torch
import datasets
from transformers import pipeline
from transformers import TrainingArguments, Trainer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
try: import evaluate
except ModuleNotFoundError:
  !pip install -U evaluate
  import evaluate

# 2. Set up variables for model training and saving pipeline
DATASET_NAME = 'mrdbourke/learn_hf_food_not_food_image_captions'
MODEL_NAME = 'distilbert/distilbert-base-uncased'
MODEL_SAVE_DIR_NAME = 'models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

# 3. Create a directory for saving models
print(f'[INFO] Creating directory for saving models: {MODEL_SAVE_DIR_NAME}')
model_save_dir = Path(MODEL_SAVE_DIR_NAME)
model_save_dir.mkdir(parents=True, exist_ok=True)

# 4. Load and preprocess dataset from Hugging Face Hub
print(f'Downloading dataset from HuggingFace Hub, name: {DATASET_NAME}')
dataset = datasets.load_dataset(DATASET_NAME)

# Use the consistent mapping: 0 for 'not_food', 1 for 'food' - CONSISTENT MAPPING
id2label = {0: 'not_food', 1: 'food'}
label2id = {'food': 1, 'not_food': 0}

# Create a function to map IDs to labels in the dataset
def map_labels_to_number(example):
  example['label'] = label2id[example['label']]
  return example

# Map our preprocessing function to our dataset
dataset = dataset['train'].map(map_labels_to_number)

# Split the dataset into train/test sets
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

# 5. Import a tokenizer and map it to our dataset
print(f'Tokenizing text for model training with tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=MODEL_NAME,
                                          use_fast=True)

# Create a function to turn text into numbers using the above tokenizer
def tokenize_text(examples):
  return tokenizer(examples['text'],
                   padding=True,
                   truncation=True)

tokenized_dataset = dataset.map(function=tokenize_text,
                                batched=True,
                                batch_size=1000)
tokenized_dataset

# 6. Set up an evaluation metric
accuracy_metric = evaluate.load('accuracy')

def compute_accuracy(predictions_and_labels):
  predictions, labels = predictions_and_labels
  # (model will output logits in the form ([[item_1, item_2, item_3], [item_1, item_2, item_3]])), depending on the number of classes you have.
  # But we want to compare labels which are in the format ([0,0,0,0,1])
  if len(predictions.shape) >= 2:
    predictions = np.argmax(predictions, axis=1) # reduce the items to a single value
  return accuracy_metric.compute(predictions=predictions, references=labels)

# 7. Set up a model
print(f'Loading model: {MODEL_NAME}')
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=2,
    id2label = id2label,
    label2id = label2id
)
print(f'Model loading complete! Model config id2label: {model.config.id2label}')

# Set up TrainingArguments (hyperparameters)
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=3,
    use_cpu=False,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy='epoch',
    report_to='none',
    push_to_hub=False,
    hub_private_repo=False
)

# Create Trainer Instance
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test'],
    processing_class=tokenizer,            # [added in]
   ### tokenizer=tokenizer,
    compute_metrics=compute_accuracy
)

trainer

# 8. Train the model
print(f'[INFO] Commencing model training...')
results = trainer.train()

# 9. Save the trained model to a local directory
print(f'[INFO] Model training complete, saving to local path: {model_save_dir}')
trainer.save_model(output_dir=model_save_dir)

# 10. Push the model to the Hugging Face hub
print(f'[INFO] Uploading model to Hugging Face Hub...')
model_upload_url = trainer.push_to_hub(
    commit_message='Uploading food not food text classifier model (putting it all together)'
)
print(f'[INFO] Model upload complete, model available at: {model_upload_url}')

# 11. Evaluate the model on the test data
print(f'[INFO] Performing evaluation on test dataset...')
predictions_all = trainer.predict(tokenized_dataset['test'])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f'[INFO] prediction metrics on test data:')
pprint.pprint(prediction_metrics)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
[INFO] Creating directory for saving models: models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/250 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

...

New Data Upload               : |          |  0.00B /  0.00B            

  ...uncased/model.safetensors:  13%|#3        | 34.8MB /  268MB            

  ...uncased/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

[INFO] Model upload complete, model available at: https://huggingface.co/ironspiritjeff/learn_hf_food_not_food_text_classifier-distilbert-base-uncased/commit/6cb9e0297dea5a448433c8d5ff143c6c4663b3cc
[INFO] Performing evaluation on test dataset...


[INFO] prediction metrics on test data:
{'test_accuracy': 1.0,
 'test_loss': 0.0004642997228074819,
 'test_runtime': 0.0929,
 'test_samples_per_second': 537.927,
 'test_steps_per_second': 21.517}


In [2]:
# 12. Make sure the model works by testing it on a custom sample
from transformers import pipeline
food_not_food_classifier = pipeline(task='text-classification',
                                    model=model_save_dir,
                                    device=torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'),
                                    top_k=1,
                                    batch_size=32
                                    )
food_not_food_classifier('after the main course we will have dessert')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[[{'label': 'not_food', 'score': 0.9987622499465942}]]

In [3]:
from typing import Dict
# import torch

# # Define the local_model_path, using the value from MODEL_SAVE_DIR_NAME if available globally
# # or explicitly setting it for self-contained execution.
# local_model_path = 'models/learn_hf_food_not_food_text_classifier-distilbert-base-uncased'

# 1. Create a function to take a string input
def food_not_food_classifier(text: str) -> Dict[str, float]:
  # Ensure local_model_path is defined within the scope of this function if it might not be global
  # local_model_path = MODEL_SAVE_DIR_NAME # Use the already defined variable from previous cells

  # 2. Set up classifier
  food_not_food_pipeline = pipeline(task='text-classification',
                                   model=model_save_dir,
                                   batch_size=32,
                                   device='cuda' if torch.cuda.is_available()else 'cpu',
                                   top_k=None)
  #3. Get outputs from pipeline
  outputs = food_not_food_pipeline(text)[0]

  # 4. Format output for Gradio
  output_dict = {}
  for item in outputs:
    output_dict[item['label']] = item['score']
  return output_dict

food_not_food_classifier(text='delicious tacos and salsa')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'food': 0.9994741082191467, 'not_food': 0.0005258601158857346}

## Building a small Gradio demo to run locally


Turning a model into a demo helps you share it with others so they can try it out and potentially give feedback for improvement.

We'll create a demo with Gradio, which helps to create the workflow.

## Creating a function to perform inference

1. Take an input of string.
2. Set up a text classification pipeline.
3. Get the output from the pipeline.
4. Return the output from the pipeline in a formatted dictionary `{'label_1': probability_1, 'label_2': probability_2}`



In [4]:
# 1. Import Gradio
import gradio as gr

# 2. Create a Gradio Interface - https://www.gradio.app/main/docs/gradio/interface
demo = gr.Interface(
    fn=food_not_food_classifier,
    inputs='text',
    outputs=gr.Label(num_top_classes=2),
    title='Food Not Food Classifier',
    description='A text classifier to determine if a sentence is about food or not.',
    examples=[['I whipped up a fresh batch of code, but it seems to have a syntax error'],
              ['A plate of pancakes and maple syrup']]
)
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://858c9646b4f4de01ad.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Making our demo publically accessible

There are 2 main ways to make the demo publically accessible on HF Spaces:

1. Manually: by going to huggingface.co/spaces -> "Create new space" -> add our files and publish.

2. Programatically: we can use the HF Hub python API and add our files to a Space with code. This requires 3 files:
* `app.py` - This is the main app with functionality of our demo.
* `requirements.txt` - these are the dependencies which our app will need.
* `README.md` - This  will explain what our project/demo is about, and add some metadata in a YAML format.

To create these 3 files to create a Space programmatically, we'll use the following file structure:
```
demos/
└── food_not_food_text_classifier/
    ├── app.py
    ├── README.md
    └── requirements.txt
```


## Making a directory to store our demo:

In [5]:
from pathlib import Path

# Make directory for demos
demos_dir = Path("../demos")
demos_dir.mkdir(exist_ok=True)

# Create a folder for the demo
food_not_food_text_classifier_demo_dir = Path(demos_dir, 'food_not_food_text_classifier')
food_not_food_text_classifier_demo_dir.mkdir(exist_ok=True)

### making an app.py file

Our app.py will contain the main logic of our application to run.

When we upload it to HF Spaces, Spaces will try to run app.py automatically.

We want the app.py file to:
1. Import packages
2. Define the function to use our model (with Gradio)
3. Create a demo with Gradio
4. Run the demo with `demo.launch()`.

To create each of our files we'll use the Ipython magic commmand `%%writefile` https://ipython.readthedocs.io/en/stable/interactive/magics.html

In [6]:
%%writefile ../demos/food_not_food_text_classifier/app.py
#1. import packages
import torch
import gradio as gr
from typing import Dict
from transformers import pipeline

# 2. Define our function to use with our model
def food_not_food_classifier(text: str) -> Dict[str, float]:
  # 2. Set up classifier
  food_not_food_pipeline = pipeline(task='text-classification',
                                   model="ironspiritjeff/learn_hf_food_not_food_text_classifier-distilbert-base-uncased",
                                    #  model=local_model_path,
                                   batch_size=32,
                                   device='cuda' if torch.cuda.is_available()else 'cpu',
                                   top_k=None) # (returns all possible labels)
  # 3. Get outputs from pipeline
  outputs = food_not_food_pipeline(text)[0]

  # 4. Format output for Gradio
  output_dict = {}
  for item in outputs:
    output_dict[item['label']] = item['score']

  return output_dict

#3. Create a Gradio interface
description = '''
A text classifier to determine if a sentence is  or isn't about food.

Fine Tuned from [DistilBERT] (https://huggingface.co/distilbert/distilbert-base-uncased) a [dataset of LLM generated food/not food image captions] (https://huggingface.co/datasets/mrdbourke/learn_hf_food_not_food_image_captions).

See [source code] (https://github.com/ironspiritjeff/hf-transformer-experiments/blob/main/HuggingFace_TextClassificationTutorial.ipynb).
'''

demo = gr.Interface(
    fn=food_not_food_classifier,
    inputs='text',
    outputs=gr.Label(num_top_classes=2),
    title='🥗🚫 Food or Not Food Text Classifier',
    description=description,
    examples=[['I whipped up a fresh batch of code, but it seems to have a syntax error'],
              ['A plate of pancakes and maple syrup']]
)

# 4.Launch the interface
if __name__ == "__main__":
  demo.launch()

Writing ../demos/food_not_food_text_classifier/app.py


### Making our README file

This file is in markdown format with a special YAML block at the top.

The YAML block is used for metadata + settings, see an example here: https://huggingface.co/docs/hub/spaces-config-reference

In [8]:
%%writefile ../demos/food_not_food_text_classifier/README.md
---
title: Food Not Food Text Classifier
emoji: 🥗🚫
colorFrom: blue
colorTo: yellow
sdk: gradio
sdk_version: 6.13.0
app_file: app.py
pinned: false
license: apache-2.0
---

# 🥗🚫 Food Not Food Text Classifier
Small demo  to showcase a text classifier to determine if a sentence is about food or not.

DistilBERT model fine-tuned on a small synthetic dataset of [250 generated food/not_food image captions](https://huggingface.co/datasets/mrdbourke/learn_hf_food_not_food_image_captions).

See [source code notebook] (https://github.com/ironspiritjeff/hf-transformer-experiments/blob/main/HuggingFace_TextClassificationTutorial.ipynb).

Overwriting ../demos/food_not_food_text_classifier/README.md


In [ ]:
%%writefile ..demos/food_not_food_text_classifier/requirements
